<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase57.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 57 FIXED — final evidence matrix, figures, tables, and manuscript-results freeze

Phase 57 adds **no new experiment**. It freezes the final computational evidence package for the manuscript.

The final study evidence is intentionally asymmetric:

- the proposed neuroimaging fidelity method has strong frozen synthetic-development, sealed-holdout, and multi-seed robustness evidence;
- an authorized real neuroimaging derivative source was normalized and represented in private FHIR R4;
- the current authorized real-source family lacks the additional source evidence needed for an honest official Arm A OMOP/MI-CDM execution;
- independent blinded external adjudication was prepared but not executed.

Phase 57 converts that evidence into publication-ready tables, figures, manuscript-safe Results language, limitations, and an explicit claim matrix.


In [1]:
# CODE CELL 1/9 — Drive, constants, helpers

import os,re,csv,json,hashlib,shutil,tempfile,zipfile,time,math
import datetime as dt
from pathlib import Path
from collections import defaultdict
import statistics

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='57_final_evidence_matrix_figures_tables_and_manuscript_results_freeze'

EXPECTED={
 'p49':('ICHI2027_Phase49_SHAREABLE','.json','f10939f00827c2f4ef4b5d9fc7ea9f8002d8ced49f33775a170dd9d502f763f9'),
 'p50':('ICHI2027_Phase50_SHAREABLE','.json','d195f42e49e7b07d8d60408e56e636e097ee40af7d5d5261733ce3b715591078'),
 'p51':('ICHI2027_Phase51_SHAREABLE','.json','31ad86c5b95cf4f6ff46f69376e75d120ec809fdcde8bd8f2ba3f4d608e257ce'),
 'p52':('ICHI2027_Phase52_SHAREABLE','.json','9750a0614b57d94ebcd68161a76ddad23743becd9aa5c99b910ffb08a74023f1'),
 'p53':('ICHI2027_Phase53_SHAREABLE','.json','4ff47277e7e3504c98659b2bab6c86dc31324bfda286b0cfc4d151b0b0fadc16'),
 'p54':('ICHI2027_Phase54_SHAREABLE','.json','76a3dfa6f1c2ece69d4f89d8db298c8f9d78befde1399875ea6b05cf8b61293d'),
 'p55':('ICHI2027_Phase55_SHAREABLE','.json','3a69736ba1b9afb42c3bf6041b6ecfa0234da438e8800478fa29dd295d80c6af'),
 'p56':('ICHI2027_Phase56_SHAREABLE','.json','44158044be4de42d99ecf44a3ae990989ff4b2be3ed3f652eacda5bb6686da51'),
}

P56_ARTIFACTS={
 'family':('ICHI2027_Phase56_FAMILY_SUMMARY','.csv','f3228089b0de8968910c70d6c9928d5bc47efba8fd5b10a399645a4460f2f50f'),
 'cohort':('ICHI2027_Phase56_COHORT_CONSEQUENCES_BY_FAMILY','.csv','6bb827e013ea83a89910c478b2ba391e0b69a90e7b299f83659734810b241a88'),
 'negative':('ICHI2027_Phase56_NEGATIVE_CONTROLS','.csv','14f8b2302ca3d489088d458befdda3d20f087c4cb72c94da5fc74b1995d9a6b7'),
 'severity':('ICHI2027_Phase56_VALUE_SEVERITY_SWEEP','.csv','bf013c244b0ac2205b657a568f58ea597e39bc70ed5d80c76c09394c2f4cd522'),
 'bootstrap':('ICHI2027_Phase56_BOOTSTRAP_CI','.csv','1a660bc12fb9ca3038314127391cad7bbed8b827d2b6dec8c4dec367019861d5'),
 'seed':('ICHI2027_Phase56_SEED_SUMMARY','.csv','f372477e89e0ee7a1875d50e36a57bf90be4ccbfad469917215c9ad68c05a244'),
}

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

def read_csv(path):
    with Path(path).open(newline='',encoding='utf-8') as f:
        return list(csv.DictReader(f))

if os.environ.get('PHASE57_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE57_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE57_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase57_'))
STAGE=WORK/'outputs'; STAGE.mkdir()

OUT_BASE=Path(os.environ.get('PHASE57_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase57')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')

print('Phase57 output:',OUT)


Mounted at /content/drive
Phase57 output: /content/drive/MyDrive/ICHI2027/Phase57/run_20261001T165802_474670Z


In [2]:
# CODE CELL 2/9 — Verify final evidence chain and Phase56 analysis artifacts

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

requirements={**EXPECTED,**P56_ARTIFACTS}
hits={k:[] for k in requirements}

for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            for key,(stem,ext,digest) in requirements.items():
                if name_matches(fn,stem,ext):
                    p=Path(here)/fn
                    try:
                        if sha256(p)==digest:
                            hits[key].append(p)
                    except OSError:
                        pass

missing=[k for k,v in hits.items() if not v]
if missing:
    raise FileNotFoundError('Missing exact checksum-approved final-evidence inputs: '+', '.join(missing))

P={k:sorted(v,key=str)[0] for k,v in hits.items()}

J={k:json.loads(P[k].read_text('utf-8')) for k in EXPECTED}

assert J['p49']['phase']=='49_comparator_role_amendment_and_executable_synthetic_results_freeze'
assert J['p49']['analysis_version']=='phase49_amended_analysis_v1.0'
print('Phase49 phase:',J['p49']['phase'])
print('Phase49 analysis version:',J['p49']['analysis_version'])
assert J['p52']['authorization_attestation_valid'] is True
assert J['p52']['real_authorized_source_normalized'] is True
assert J['p53']['all_structural_preflight_checks_passed'] is True
assert J['p54']['dimensions_with_joinable_candidate_evidence']==0
assert J['p55']['independently_blinded_external_validation'] is False
assert J['p56']['phase43_truth_seal_reproduced'] is True
assert J['p56']['method_retuned'] is False
assert J['p56']['exact_case_match_rate']==1.0
assert J['p56']['unexpected_alerts']==0
assert J['p56']['negative_control_failures']==0
assert J['p56']['deterministic_reproducibility_passed'] is True

family=read_csv(P['family'])
cohort=read_csv(P['cohort'])
negative=read_csv(P['negative'])
severity=read_csv(P['severity'])
bootstrap=read_csv(P['bootstrap'])
seed_summary=read_csv(P['seed'])

print('PASS: complete Phase49→56 evidence chain and exact Phase56 analysis artifacts verified.')


Phase49 phase: 49_comparator_role_amendment_and_executable_synthetic_results_freeze
Phase49 analysis version: phase49_amended_analysis_v1.0
PASS: complete Phase49→56 evidence chain and exact Phase56 analysis artifacts verified.


In [3]:
# CODE CELL 3/9 — Freeze final evidence matrix

evidence_matrix=[
 {
  'evidence_domain':'Method freeze',
  'phase':'42',
  'status':'SUPPORTED',
  'evidence':'Frozen neuroimaging rule set and tolerance; no retuning in later analyses.',
  'allowed_claim':'Method specification was frozen before the held-out and multi-seed robustness evaluations.',
  'prohibited_overclaim':'Prospective clinical validation.'
 },
 {
  'evidence_domain':'Developer synthetic challenge',
  'phase':'42',
  'status':'SUPPORTED_WITH_SCOPE',
  'evidence':'195 authored developer fault events matched; 0 unexpected alerts; 15 valid normalizations.',
  'allowed_claim':'The frozen rules detected the authored developer challenge events in synthetic source projections.',
  'prohibited_overclaim':'Population sensitivity, specificity, or PPV.'
 },
 {
  'evidence_domain':'Untouched synthetic holdout',
  'phase':'43',
  'status':'SUPPORTED_WITH_SCOPE',
  'evidence':'229 cases; 240 authored fault events; truth sealed before detection; no method retuning.',
  'allowed_claim':'Performance reproduced on a held-out synthetic challenge generated after method freeze.',
  'prohibited_overclaim':'Independent external validation.'
 },
 {
  'evidence_domain':'Multi-seed robustness',
  'phase':'56',
  'status':'SUPPORTED_WITH_SCOPE',
  'evidence':f"{J['p56']['new_synthetic_cases']} new cases across {J['p56']['robustness_seed_count']} predeclared seeds; "
             f"{J['p56']['matched_fault_events']}/{J['p56']['authored_fault_events']} authored events matched; "
             f"{J['p56']['unexpected_alerts']} unexpected alerts.",
  'allowed_claim':'The frozen method was stable across repeated synthetic challenge realizations.',
  'prohibited_overclaim':'External replication or real-world prevalence performance.'
 },
 {
  'evidence_domain':'Negative controls and normalization',
  'phase':'56',
  'status':'SUPPORTED_WITH_SCOPE',
  'evidence':f"{J['p56']['negative_control_cases']} negative-control cases with "
             f"{J['p56']['negative_control_failures']} failures; "
             f"{J['p56']['valid_normalizations_matched']}/{J['p56']['valid_normalizations_expected']} valid normalizations matched.",
  'allowed_claim':'Predeclared negative controls and valid-normalization cases behaved as expected.',
  'prohibited_overclaim':'Universal absence of false positives.'
 },
 {
  'evidence_domain':'Deterministic reproducibility',
  'phase':'56',
  'status':'SUPPORTED',
  'evidence':'Canonical compact and full-detail rerun hashes reproduced.',
  'allowed_claim':'The computational robustness run was deterministically reproducible for the rerun seed.',
  'prohibited_overclaim':'Cross-platform reproducibility unless separately demonstrated.'
 },
 {
  'evidence_domain':'Authorized real neuroimaging source',
  'phase':'52',
  'status':'SUPPORTED_WITH_SCOPE',
  'evidence':f"{J['p52']['source_unique_persons']} authorized source persons; "
             f"{J['p52']['normalized_longitudinal_measurement_rows']} normalized longitudinal measurement rows; "
             f"{J['p52']['persons_complete_bilateral_hippocampus_baseline_m12']} persons complete for bilateral hippocampus baseline→M12.",
  'allowed_claim':'An authorized real derivative source was normalized into an immutable private longitudinal evidence reference.',
  'prohibited_overclaim':'Clinical MCI eligibility or official target fidelity performance.'
 },
 {
  'evidence_domain':'Private FHIR source representation',
  'phase':'53',
  'status':'SUPPORTED_WITH_SCOPE',
  'evidence':f"{J['p53']['fhir_patient_count']} private FHIR Patients and "
             f"{J['p53']['fhir_observation_count']} Observations; structural preflight passed; missing fields were not invented.",
  'allowed_claim':'The authorized normalized source could be represented deterministically in private FHIR R4 with explicit missingness.',
  'prohibited_overclaim':'Official FHIR profile conformance beyond the stated structural checks.'
 },
 {
  'evidence_domain':'Official Arm A OMOP/MI-CDM execution',
  'phase':'54',
  'status':'NOT_SUPPORTED',
  'evidence':'No joinable companion evidence was found for demographics, units, acquisition identifiers, model/version, provenance, or diagnosis.',
  'allowed_claim':'The current authorized source family was insufficient for an honest official Arm A execution.',
  'prohibited_overclaim':'Official OMOP ETL, official MI-CDM load, or real-data source→target fidelity.'
 },
 {
  'evidence_domain':'Independent blinded external validation',
  'phase':'55/56',
  'status':'NOT_SUPPORTED',
  'evidence':'A blinded package was prepared, but no external reviewer response was obtained.',
  'allowed_claim':'External adjudication was prepared but not executed.',
  'prohibited_overclaim':'Independent external validation.'
 }
]

evidence_path=STAGE/'ICHI2027_Phase57_FINAL_EVIDENCE_MATRIX.csv'
write_csv(evidence_path,evidence_matrix)

print('PASS: final evidence matrix frozen.')


PASS: final evidence matrix frozen.


In [4]:
# CODE CELL 4/9 — Final manuscript tables

table1=[
 {'item':'Authorized real source persons','value':J['p52']['source_unique_persons'],'scope':'real authorized source'},
 {'item':'Normalized longitudinal measurement rows','value':J['p52']['normalized_longitudinal_measurement_rows'],'scope':'real authorized source'},
 {'item':'Persons complete bilateral hippocampus baseline→M12','value':J['p52']['persons_complete_bilateral_hippocampus_baseline_m12'],'scope':'real authorized source'},
 {'item':'Private FHIR Patient resources','value':J['p53']['fhir_patient_count'],'scope':'private FHIR representation'},
 {'item':'Private FHIR Observation resources','value':J['p53']['fhir_observation_count'],'scope':'private FHIR representation'},
 {'item':'Multi-seed robustness cases','value':J['p56']['new_synthetic_cases'],'scope':'synthetic robustness'},
 {'item':'Authored fault events','value':J['p56']['authored_fault_events'],'scope':'synthetic robustness'},
 {'item':'Matched fault events','value':J['p56']['matched_fault_events'],'scope':'synthetic robustness'},
 {'item':'Unexpected alerts','value':J['p56']['unexpected_alerts'],'scope':'synthetic robustness'},
 {'item':'Negative-control cases','value':J['p56']['negative_control_cases'],'scope':'synthetic robustness'},
 {'item':'Negative-control failures','value':J['p56']['negative_control_failures'],'scope':'synthetic robustness'},
 {'item':'Valid normalizations expected/matched','value':f"{J['p56']['valid_normalizations_matched']}/{J['p56']['valid_normalizations_expected']}",'scope':'synthetic robustness'},
]

table1_path=STAGE/'ICHI2027_Phase57_TABLE1_STUDY_EVIDENCE_OVERVIEW.csv'
write_csv(table1_path,table1)

# Compact family table with only publication-useful metrics.
table2=[]
for r in family:
    table2.append({
      'case_kind':r['case_kind'],
      'cases':r['cases'],
      'truth_events':r['truth_events'],
      'matched_truth_events':r['matched_truth_events'],
      'unexpected_alerts':r['unexpected_alerts'],
      'event_recall':r['event_recall'],
      'exact_case_match_rate':r['exact_case_match_rate']
    })
table2_path=STAGE/'ICHI2027_Phase57_TABLE2_ROBUSTNESS_BY_FAULT_FAMILY.csv'
write_csv(table2_path,table2)

table3=[]
for r in cohort:
    table3.append({
      'case_kind':r['case_kind'],
      'cases':r['cases'],
      'false_exclusion_case_rate':r['false_exclusion_case_rate'],
      'false_inclusion_case_rate':r['false_inclusion_case_rate'],
      'false_exclusions_total':r['false_exclusions_total'],
      'false_inclusions_total':r['false_inclusions_total'],
      'mean_jaccard':r['mean_jaccard'],
      'min_jaccard':r['min_jaccard']
    })
table3_path=STAGE/'ICHI2027_Phase57_TABLE3_COHORT_CONSEQUENCES_BY_FAULT_FAMILY.csv'
write_csv(table3_path,table3)

table4_path=STAGE/'ICHI2027_Phase57_TABLE4_BOOTSTRAP_UNCERTAINTY.csv'
write_csv(table4_path,bootstrap)

print('PASS: final manuscript tables frozen.')


PASS: final manuscript tables frozen.


In [5]:
# CODE CELL 5/9 — Publication figures

import matplotlib.pyplot as plt

# Figure 1 — exact case agreement by challenge family
fig1=STAGE/'ICHI2027_Phase57_FIGURE1_EXACT_CASE_MATCH_BY_FAMILY.png'
labels=[r['case_kind'] for r in table2]
vals=[float(r['exact_case_match_rate']) for r in table2]

plt.figure(figsize=(10,7))
plt.barh(labels,vals)
plt.xlim(0,1.02)
plt.xlabel('Exact case-level agreement')
plt.ylabel('Challenge family')
plt.title('Frozen-method agreement across multi-seed synthetic robustness challenges')
plt.tight_layout()
plt.savefig(fig1,dpi=300,bbox_inches='tight')
plt.close()

# Figure 2 — downstream cohort consequences by family
fig2=STAGE/'ICHI2027_Phase57_FIGURE2_COHORT_FALSE_EXCLUSION_RATE.png'
labels2=[r['case_kind'] for r in table3]
vals2=[float(r['false_exclusion_case_rate']) for r in table3]

plt.figure(figsize=(10,7))
plt.barh(labels2,vals2)
plt.xlim(0,1.02)
plt.xlabel('Cases causing ≥1 false exclusion')
plt.ylabel('Challenge family')
plt.title('Downstream cohort consequences of synthetic transformation faults')
plt.tight_layout()
plt.savefig(fig2,dpi=300,bbox_inches='tight')
plt.close()

# Figure 3 — frozen numeric tolerance severity characterization
fig3=STAGE/'ICHI2027_Phase57_FIGURE3_VALUE_SEVERITY_SWEEP.png'
x=[float(r['absolute_delta_mm3']) for r in severity]
y=[float(r['detection_rate']) for r in severity]

plt.figure(figsize=(8,5))
plt.plot(x,y,marker='o')
plt.axvline(J['p56']['frozen_abs_tolerance_mm3'],linestyle='--')
plt.xscale('symlog',linthresh=1e-7)
plt.ylim(-0.02,1.02)
plt.xlabel('Absolute numeric perturbation (mm³)')
plt.ylabel('VALUE_MISMATCH detection rate')
plt.title('Characterization of the pre-existing frozen numeric tolerance')
plt.tight_layout()
plt.savefig(fig3,dpi=300,bbox_inches='tight')
plt.close()

# Figure 4 — bootstrap intervals
fig4=STAGE/'ICHI2027_Phase57_FIGURE4_BOOTSTRAP_INTERVALS.png'
metrics=[r['metric'] for r in bootstrap]
est=[float(r['estimate']) for r in bootstrap]
lo=[float(r['ci95_lower']) for r in bootstrap]
hi=[float(r['ci95_upper']) for r in bootstrap]
ypos=list(range(len(metrics)))
lower=[e-l for e,l in zip(est,lo)]
upper=[h-e for e,h in zip(est,hi)]

plt.figure(figsize=(8,5))
plt.errorbar(est,ypos,xerr=[lower,upper],fmt='o',capsize=4)
plt.yticks(ypos,metrics)
plt.xlabel('Estimate with 95% seed-cluster bootstrap interval')
plt.title('Uncertainty across predeclared robustness seeds')
plt.tight_layout()
plt.savefig(fig4,dpi=300,bbox_inches='tight')
plt.close()

for p in [fig1,fig2,fig3,fig4]:
    assert p.is_file() and p.stat().st_size>0

print('PASS: four publication figures created.')


PASS: four publication figures created.


In [6]:
# CODE CELL 6/9 — Freeze manuscript-safe Results text

results_text=f"""RESULTS — FROZEN MANUSCRIPT-SAFE VERSION

Method freeze and synthetic validation.
The neuroimaging fidelity rule set was frozen before held-out evaluation and was not retuned thereafter. The original sealed holdout contained 229 synthetic cases and 240 authored fault events. The Phase 43 truth seal was reproduced before the final robustness extension.

Multi-seed robustness.
The frozen method was evaluated on {J['p56']['new_synthetic_cases']:,} newly generated synthetic cases across {J['p56']['robustness_seed_count']} predeclared random seeds. These cases contained {J['p56']['authored_fault_events']:,} authored fault events. The detector matched {J['p56']['matched_fault_events']:,}/{J['p56']['authored_fault_events']:,} authored events and generated {J['p56']['unexpected_alerts']} unexpected alerts. Exact case-level agreement was {J['p56']['exact_case_matches']:,}/{J['p56']['new_synthetic_cases']:,} ({100*J['p56']['exact_case_match_rate']:.1f}%). The analysis included {J['p56']['negative_control_cases']} negative-control cases, with {J['p56']['negative_control_failures']} failures, and {J['p56']['valid_normalizations_matched']}/{J['p56']['valid_normalizations_expected']} valid unit-normalization cases were recognized as expected. The existing absolute value tolerance of {J['p56']['frozen_abs_tolerance_mm3']:g} mm3 was characterized across {J['p56']['severity_sweep_levels']} predeclared perturbation levels without changing the threshold. Uncertainty was estimated with {J['p56']['bootstrap_reps']:,} bootstrap resamples clustered by robustness seed. A deterministic rerun reproduced the canonical analysis hashes.

Authorized real-source feasibility.
A private authorized longitudinal neuroimaging derivative source included {J['p52']['source_unique_persons']} persons and {J['p52']['normalized_longitudinal_measurement_rows']:,} normalized longitudinal measurement rows. All {J['p52']['persons_complete_bilateral_hippocampus_baseline_m12']} persons had bilateral hippocampal measurements at baseline and approximately 12 months in the wide source schema. This source was converted to a private normalized longitudinal reference and then represented as {J['p53']['fhir_patient_count']} FHIR Patient resources and {J['p53']['fhir_observation_count']:,} FHIR Observation resources. Structural preflight checks passed, and source information that was not available—including explicit units, acquisition identifiers, model version, provenance identifiers, and clinical MCI status—was not fabricated.

Official-target feasibility.
The authorized source family did not contain joinable companion evidence for the missing demographic, unit, acquisition, model/version, provenance, or diagnostic dimensions. Therefore, an official Arm A OMOP/MI-CDM transformation was not executed, and the real-data source was not processed through the proposed source-to-target fidelity method.

External validation.
A blinded external-adjudication package was prepared from the sealed synthetic holdout, but no independent reviewer response was obtained. Independent blinded external validation was therefore not performed. The final validation scope is frozen-method multi-seed synthetic robustness and reproducibility, not external replication or clinical validation.
"""

results_path=STAGE/'ICHI2027_Phase57_MANUSCRIPT_RESULTS_FROZEN.txt'
results_path.write_text(results_text,encoding='utf-8')

print(results_text)


RESULTS — FROZEN MANUSCRIPT-SAFE VERSION

Method freeze and synthetic validation.
The neuroimaging fidelity rule set was frozen before held-out evaluation and was not retuned thereafter. The original sealed holdout contained 229 synthetic cases and 240 authored fault events. The Phase 43 truth seal was reproduced before the final robustness extension.

Multi-seed robustness.
The frozen method was evaluated on 2,748 newly generated synthetic cases across 12 predeclared random seeds. These cases contained 2,880 authored fault events. The detector matched 2,880/2,880 authored events and generated 0 unexpected alerts. Exact case-level agreement was 2,748/2,748 (100.0%). The analysis included 48 negative-control cases, with 0 failures, and 180/180 valid unit-normalization cases were recognized as expected. The existing absolute value tolerance of 1e-06 mm3 was characterized across 8 predeclared perturbation levels without changing the threshold. Uncertainty was estimated with 5,000 bootstra

In [7]:
# CODE CELL 7/9 — Freeze limitations and claim-boundary text

limitations_text="""LIMITATIONS — FROZEN MANUSCRIPT-SAFE VERSION

The strongest quantitative evaluation in this study is synthetic. Although the method was frozen before held-out evaluation, reproduced the original truth seal, remained stable across 12 additional predeclared seeds, passed negative controls, and was deterministically reproducible, repeated synthetic robustness does not establish independent external validity or real-world prevalence performance.

The authorized real neuroimaging derivative source supported longitudinal bilateral hippocampal evidence and deterministic private FHIR representation, but it did not provide explicit measurement units, acquisition or DICOM identifiers, extraction-model version, provenance identifiers, or clinical MCI status. A bounded search of the nearby authorized source family did not identify joinable companion evidence for those dimensions. Consequently, an official OMOP/MI-CDM Arm A transformation and real-data source-to-target fidelity experiment were not executed.

A blinded external-adjudication package was prepared, but an eligible independent reviewer did not complete the adjudication. Independent external validation therefore remains future work.

The results should be interpreted as evidence that the proposed loss-aware rules can reproducibly detect deliberately authored fidelity failures in the defined synthetic challenge framework and can preserve explicit missingness when constructing a private FHIR representation of an authorized real derivative source. The study does not establish clinical safety, diagnostic validity, official OMOP/MI-CDM conformance, or generalization to other institutions, imaging pipelines, or source schemas.
"""

claims=[
 {
  'claim':'The method was frozen before holdout and robustness evaluation.',
  'status':'SUPPORTED',
  'basis':'Phase42 method hash + Phase43/56 no-retuning checks.'
 },
 {
  'claim':'The frozen method was robust across repeated synthetic challenge realizations.',
  'status':'SUPPORTED_WITH_SYNTHETIC_SCOPE',
  'basis':f"{J['p56']['new_synthetic_cases']} cases, {J['p56']['robustness_seed_count']} seeds, exact case agreement {J['p56']['exact_case_match_rate']:.3f}."
 },
 {
  'claim':'The authorized real derivative source can be represented in private FHIR with explicit missingness.',
  'status':'SUPPORTED_WITH_SOURCE_SCOPE',
  'basis':'Phase52 normalization + Phase53 deterministic private FHIR representation.'
 },
 {
  'claim':'The method has been independently externally validated.',
  'status':'NOT_SUPPORTED',
  'basis':'No external reviewer response.'
 },
 {
  'claim':'The method has been validated on a real OMOP/MI-CDM source-to-target transformation.',
  'status':'NOT_SUPPORTED',
  'basis':'Official Arm A not executable from current authorized source family.'
 },
 {
  'claim':'The reported synthetic rates are clinical sensitivity/specificity/PPV.',
  'status':'NOT_SUPPORTED',
  'basis':'Authored synthetic challenge design is not a population sample.'
 },
 {
  'claim':'The method is clinically safe or generalizes across institutions.',
  'status':'NOT_SUPPORTED',
  'basis':'No operational clinical or multi-institutional validation.'
 }
]

limitations_path=STAGE/'ICHI2027_Phase57_LIMITATIONS_FROZEN.txt'
limitations_path.write_text(limitations_text,encoding='utf-8')

claims_path=STAGE/'ICHI2027_Phase57_FINAL_CLAIM_MATRIX.csv'
write_csv(claims_path,claims)

print('PASS: limitations and final claim boundary frozen.')


PASS: limitations and final claim boundary frozen.


In [8]:
# CODE CELL 8/9 — Figure/table captions and manuscript mapping

mapping=[
 {
  'artifact':'TABLE1_STUDY_EVIDENCE_OVERVIEW',
  'manuscript_role':'Results overview',
  'caption':'Summary of the authorized real-source feasibility evidence and frozen synthetic robustness evaluation.'
 },
 {
  'artifact':'TABLE2_ROBUSTNESS_BY_FAULT_FAMILY',
  'manuscript_role':'Primary synthetic robustness table',
  'caption':'Frozen-method performance across predeclared synthetic challenge families aggregated over 12 robustness seeds.'
 },
 {
  'artifact':'TABLE3_COHORT_CONSEQUENCES_BY_FAULT_FAMILY',
  'manuscript_role':'Downstream consequence table',
  'caption':'Effect of synthetic transformation faults on the predefined downstream cohort proxy.'
 },
 {
  'artifact':'TABLE4_BOOTSTRAP_UNCERTAINTY',
  'manuscript_role':'Uncertainty table',
  'caption':'Seed-cluster bootstrap estimates and 95% intervals for the principal robustness metrics.'
 },
 {
  'artifact':'FIGURE1_EXACT_CASE_MATCH_BY_FAMILY',
  'manuscript_role':'Primary robustness figure',
  'caption':'Exact case-level agreement of the frozen method across multi-seed synthetic challenge families.'
 },
 {
  'artifact':'FIGURE2_COHORT_FALSE_EXCLUSION_RATE',
  'manuscript_role':'Outcome consequence figure',
  'caption':'Fraction of synthetic challenge cases producing at least one false exclusion from the predefined cohort proxy.'
 },
 {
  'artifact':'FIGURE3_VALUE_SEVERITY_SWEEP',
  'manuscript_role':'Threshold characterization',
  'caption':'Detection behavior around the pre-existing frozen absolute numeric tolerance; the threshold was characterized but not tuned.'
 },
 {
  'artifact':'FIGURE4_BOOTSTRAP_INTERVALS',
  'manuscript_role':'Uncertainty figure',
  'caption':'Seed-cluster bootstrap uncertainty for principal multi-seed robustness metrics.'
 }
]

mapping_path=STAGE/'ICHI2027_Phase57_FIGURE_TABLE_MANUSCRIPT_MAP.csv'
write_csv(mapping_path,mapping)

print('PASS: figure/table manuscript map frozen.')


PASS: figure/table manuscript map frozen.


In [9]:
# CODE CELL 9/9 — Final package, README, and Phase57 SHAREABLE

readme=STAGE/'ICHI2027_Phase57_README.txt'
readme.write_text(
    'Phase57 is the final evidence/results freeze and introduces no new experiment.\n'
    'It verifies the Phase49-56 evidence chain, freezes publication tables and figures, and creates manuscript-safe Results, limitations, and claim-boundary text.\n'
    'The quantitative validation claim is frozen-method multi-seed synthetic robustness and reproducibility.\n'
    'Authorized real-source feasibility and private FHIR representation are documented separately from source-to-target fidelity performance.\n'
    'Independent external validation, official OMOP ETL, official MI-CDM loading, and real-data source-to-target fidelity remain unperformed and must not be claimed.\n',
    encoding='utf-8'
)

artifacts=[
    evidence_path,
    table1_path,table2_path,table3_path,table4_path,
    fig1,fig2,fig3,fig4,
    results_path,limitations_path,claims_path,mapping_path,readme
]

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'input_phase49_shareable_sha256':sha256(P['p49']),
 'input_phase50_shareable_sha256':sha256(P['p50']),
 'input_phase51_shareable_sha256':sha256(P['p51']),
 'input_phase52_shareable_sha256':sha256(P['p52']),
 'input_phase53_shareable_sha256':sha256(P['p53']),
 'input_phase54_shareable_sha256':sha256(P['p54']),
 'input_phase55_shareable_sha256':sha256(P['p55']),
 'input_phase56_shareable_sha256':sha256(P['p56']),
 'final_evidence_matrix_frozen':True,
 'final_tables_created':4,
 'final_figures_created':4,
 'manuscript_results_frozen':True,
 'limitations_frozen':True,
 'final_claim_matrix_frozen':True,
 'method_retuned':False,
 'phase43_truth_seal_reproduced':True,
 'multiseed_synthetic_cases':J['p56']['new_synthetic_cases'],
 'multiseed_authored_fault_events':J['p56']['authored_fault_events'],
 'multiseed_matched_fault_events':J['p56']['matched_fault_events'],
 'multiseed_unexpected_alerts':J['p56']['unexpected_alerts'],
 'authorized_real_source_persons':J['p52']['source_unique_persons'],
 'private_fhir_patients':J['p53']['fhir_patient_count'],
 'private_fhir_observations':J['p53']['fhir_observation_count'],
 'official_omop_etl_executed':False,
 'official_micdm_load_executed':False,
 'real_authorized_mri_processed_by_proposed_fidelity_method':False,
 'independently_blinded_external_validation':False,
 'final_validation_scope':'FROZEN_METHOD_SYNTHETIC_HOLDOUT_PLUS_MULTI_SEED_ROBUSTNESS_WITH_AUTHORIZED_REAL_SOURCE_FHIR_FEASIBILITY',
 'experimental_work_status':'FROZEN_NO_ADDITIONAL_EXPERIMENT_PLANNED_FOR_CURRENT_MANUSCRIPT',
 'next_step':'MANUSCRIPT_ASSEMBLY_AND_SUBMISSION_PACKAGE',
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'Final evidence freeze does not convert unperformed official-target or external-validation analyses into supported claims.'
}

summary_path=STAGE/'ICHI2027_Phase57_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

public_zip=STAGE/'ICHI2027_Phase57_FINAL_RESULTS_PACKAGE.zip'
with zipfile.ZipFile(public_zip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:
        z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[public_zip]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2:
                raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase57 final evidence/results freeze completed.')
print('Experiments: FROZEN')
print('Final tables:',4)
print('Final figures:',4)
print('Next step:',summary['next_step'])
print('Shareable:',OUT/summary_path.name)
print('Final results ZIP:',OUT/public_zip.name)


SUCCESS: Phase57 final evidence/results freeze completed.
Experiments: FROZEN
Final tables: 4
Final figures: 4
Next step: MANUSCRIPT_ASSEMBLY_AND_SUBMISSION_PACKAGE
Shareable: /content/drive/MyDrive/ICHI2027/Phase57/run_20261001T165802_474670Z/ICHI2027_Phase57_SHAREABLE.json
Final results ZIP: /content/drive/MyDrive/ICHI2027/Phase57/run_20261001T165802_474670Z/ICHI2027_Phase57_FINAL_RESULTS_PACKAGE.zip


## After Phase 57

No additional experiment is planned for the current manuscript.

Upload **`ICHI2027_Phase57_SHAREABLE.json`**. The next work should be manuscript assembly: Introduction/Methods alignment, Results insertion from the frozen text, figures/tables placement, Discussion, limitations, abstract, and submission checklist.

Any future official OMOP/MI-CDM execution or external validation should be treated as a later validation extension rather than silently changing this frozen evidence package.
